# RLVR with a Custom Reward Function

## Lab 4 - Evaluate the customized model

In this lab, you will evaluate the RLVR fine-tuned model against the **MATH benchmark** to measure its mathematical reasoning.

### Why benchmark evaluation?

RLVR trains models on tasks with objectively verifiable answers, so it makes sense to evaluate them the same way — using a standardized benchmark with known correct answers rather than subjective LLM-as-a-Judge scoring.

### What you'll do in this notebook

1. Retrieve the fine-tuned model from the Model Registry
2. Explore available benchmarks and create a `BenchMarkEvaluator`
3. Run the MATH benchmark evaluation on the fine-tuned model
4. View the results

---

## 1. Prerequisites and SageMaker setup

In [ ]:
%pip install -r requirements.txt

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

base_model_id = "huggingface-reasoning-qwen3-06b"
project_prefix = "gsm8k-custom-reward-rlvr"

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

---

## 2. Retrieve the custom-reward fine-tuned model

This mirrors Lab 3a: we locate the latest model package in the model package group created by the training notebook. If you want to evaluate a specific model package version, set `fine_tuned_model_package_arn` manually before running the lookup cell.

In [ ]:
import hashlib

MAX_MPG_NAME_LENGTH = 63
suffix = "-custom-rw-rlvr"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    # reserve room for the suffix, a hyphen separator, and the 6-char hash
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

In [ ]:
from sagemaker.core.resources import ModelPackageGroup

response = sm_client.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1,
)

if len(response["ModelPackageSummaryList"]) > 0:
    fine_tuned_model_package_arn = response["ModelPackageSummaryList"][0]["ModelPackageArn"]
    fine_tuned_model_package_group_arn = ModelPackageGroup.get(model_package_group_name).model_package_group_arn
else:
    fine_tuned_model_package_arn = None
    fine_tuned_model_package_group_arn = None

if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{project_prefix}/evaluation"
else:
    output_path = f"s3://{bucket_name}/{project_prefix}/evaluation"

print(f"Model Package Group: {model_package_group_name}")
print(f"Model Package Group ARN: {fine_tuned_model_package_group_arn}")
print(f"Fine-tuned Model Package ARN: {fine_tuned_model_package_arn}")
print(f"Evaluation output path: {output_path}")

---

## 3. Explore available benchmarks

SageMaker provides several built-in benchmarks. As in Lab 3a, this lab uses the **MATH** benchmark for mathematical reasoning evaluation.

In [ ]:
from sagemaker.train.evaluate import BenchMarkEvaluator, get_benchmarks, get_benchmark_properties
from rich.pretty import pprint

Benchmark = get_benchmarks()
pprint(list(Benchmark))

In [ ]:
pprint(get_benchmark_properties(benchmark=Benchmark.MATH))

---

## 4. Run the evaluation, or use pre-computed results

This follows the same `BenchMarkEvaluator` pattern as Lab 3a: pass the benchmark enum
(`Benchmark.MATH`) and use `s3_output_path` for the results.

The MATH benchmark takes **15-30 minutes**. You can either run it yourself or load a
result produced by exactly the cell below, so the rest of the notebook is identical
either way.

Set the switch in the next cell once. Everything after it follows that setting, so
**Run All** works whichever you choose.

In [ ]:
# The one switch in this notebook.
#   False -> load pre-computed results, about a minute
#   True  -> run the evaluations yourself
RUN_EVALUATIONS = False

if RUN_EVALUATIONS:
    print("Will launch the MATH benchmark job for the fine-tuned model.")
    print("Expect 15-30 minutes. The notebook blocks until it finishes.")
else:
    print("Will load a pre-computed result from the workshop asset bucket.")
    print("No jobs are launched and nothing is billed.")
    print("Set RUN_EVALUATIONS = True above to run the evaluations instead.")

In [ ]:
import json
import os

import boto3

from sagemaker.train.common_utils.show_results_utils import (
    _display_metrics_tables,
    _extract_metrics_from_results,
)

if RUN_EVALUATIONS:
    evaluator = BenchMarkEvaluator(
        benchmark=Benchmark.MATH,
        model=fine_tuned_model_package_arn,
        model_package_group=model_package_group_name,
        base_eval_name="custom-reward-rlvr",
        s3_output_path=output_path,
        evaluate_base_model=False,
        role=role,
        sagemaker_session=sess,
    )

    execution = evaluator.evaluate()
    execution.wait()
    print(execution)
else:
    os.makedirs("./eval_results", exist_ok=True)

    PRECOMPUTED_BUCKET = "ws-assets-prod-iad-r-iad-ed304a55c2ca1aee"
    PRECOMPUTED_PREFIX = "548b5be9-2da8-4c93-82f7-b0b474108ab3/lab3a/eval_results"

    s3 = boto3.client("s3", region_name="us-east-1")
    s3.download_file(PRECOMPUTED_BUCKET, f"{PRECOMPUTED_PREFIX}/tuned_benchmark_results.json",
                     "./eval_results/tuned_benchmark_results.json")

    with open("./eval_results/tuned_benchmark_results.json") as f:
        tuned_results = json.load(f)
    print("downloaded:", sorted(os.listdir("./eval_results")))

---

## 5. View evaluation results

The MATH benchmark scores for the RLVR fine-tuned model. On the live path these come
from the evaluation execution; on the pre-computed path they come from the file
downloaded above. The table is the same in both cases.

In [ ]:
from rich.pretty import pprint

from sagemaker.train.evaluate import EvaluationPipelineExecution
from sagemaker.train.evaluate.constants import EvalType

if RUN_EVALUATIONS:
    # This lab runs a single benchmark evaluation, so we display the one we just
    # launched. Prefer the `execution` object returned by evaluator.evaluate() above;
    # fall back to looking it up by its S3 output path (get_all() returns executions
    # from every benchmark pipeline in the account and is not ordered by time, so a
    # plain [-1] could pick an unrelated run).
    try:
        result_execution = execution
    except NameError:
        result_execution = next(
            (
                e
                for e in EvaluationPipelineExecution.get_all(eval_type=EvalType.BENCHMARK)
                if e.status.overall_status == "Succeeded"
                and getattr(e, "s3_output_path", None) == output_path
            ),
            None,
        )

    pprint(result_execution)
    pprint(result_execution.show_results())
else:
    # the renderer show_results() uses, fed from disk instead of a live execution.
    # This lab evaluates the tuned model only, so there is no base column to show.
    _display_metrics_tables(
        _extract_metrics_from_results(tuned_results),
        None,
        {"custom": f"s3://{PRECOMPUTED_BUCKET}/{PRECOMPUTED_PREFIX}/", "base": None},
    )